# Libraries

In [0]:
# Installing new libraries
%pip install -r ../requirements.txt
%restart_python

In [0]:
# Importing required libraries
import numpy as np
import pandas as pd

# Data

## Reading

In [0]:
# Querying transaction table to get LBE transactions
def get_transactions():
    df = spark.sql(
        """
        SELECT 
            DATE_SUB(DATE(TRANSACTION_DATETIME), (DAYOFWEEK(TRANSACTION_DATETIME) + 1) % 7) AS week,
            CAST(SUM(QUANTITY) AS INT) AS transactions,
            ROUND(SUM(NET_REVENUE), 2) AS revenue
        FROM CPX_DATAANALYTICS_GOLD.CUSTOMER360.C360_F_TRANSACTION
        WHERE TRANSACTION_DATETIME BETWEEN '2023-03-31' AND '2025-07-03'
        AND (QUANTITY - CAST(QUANTITY AS INT)) = 0
        AND LOCATIONID IN (
            SELECT LOCATIONID
            FROM CPX_DATAANALYTICS_GOLD.CPX_DW.DIM_LOCATION
            WHERE LC_TRADE_BRAND_DESCRIPTION IN ('Playdium', 'Rec Room')
            AND STATUS <> 'Closed'
        )
        GROUP BY DATE_SUB(DATE(TRANSACTION_DATETIME), (DAYOFWEEK(TRANSACTION_DATETIME) + 1) % 7)
        ORDER BY week
        """
    ).toPandas()

    return df

In [0]:
# Setting raw data path
directory = '/Volumes/training_feature_development/marketing_mix_modelling/data_v2/'

# Reading spend data
sem = pd.read_excel(directory + 'raw_data/SEM EXH & LBE Mar 2023 to Jun 2025 weekly report_v2.xlsx', skiprows=2)
datorama = pd.read_excel(directory + 'raw_data/Datorama Report All Channels Jan 2023 to Apr 2025 Daily Report.xlsx', skiprows=4, skipfooter=1)
pr = pd.read_csv(directory + 'raw_data/pr.csv')

# Reading other data
transactions = get_transactions() # Backup in 'lbe_transactions.csv'
recroom_web = pd.read_csv(directory + 'raw_data/TheRecRoom.com - Day.csv', skiprows=11)
playdium_web = pd.read_csv(directory + 'raw_data/Playdium Production - Day.csv', skiprows=11)
mass_offers = pd.read_csv(directory + 'raw_data/mass_offers.csv')

## Cleaning

In [0]:
# Replacing headers
playdium_web.columns = ['date', 'pdm_web_visits', 'unique_visitors', 'orders']
recroom_web.columns = ['date', 'trr_web_visits', 'unique_visitors', 'orders']

# Cleaning column names
for df in [sem, datorama, pr, transactions, recroom_web, playdium_web, mass_offers]:
  df.columns = df.columns.str.lower().str.replace(' ', '_')

# Creating new columns
pr['spend'] = pr[pr.columns[-3:]].sum(axis=1)
mass_offers['mass_offer'] = mass_offers.max(axis=1, numeric_only=True)

# Making data frequency weekly, starting Friday
sem['week'] = pd.to_datetime(sem['week']).dt.to_period('W-THU').dt.start_time
datorama['week'] = pd.to_datetime(datorama['day']).dt.to_period('W-THU').dt.start_time
pr['week'] = pd.to_datetime(pr['week']).dt.to_period('W-THU').dt.start_time
transactions['week'] = pd.to_datetime(transactions['week'])
recroom_web['week'] = pd.to_datetime(recroom_web['date']).dt.to_period('W-THU').dt.start_time
playdium_web['week'] = pd.to_datetime(playdium_web['date']).dt.to_period('W-THU').dt.start_time
mass_offers['week'] = pd.to_datetime(mass_offers['week']).dt.to_period('W-THU').dt.start_time

# Splitting Datorama dataframe
meta = datorama[datorama['publisher']=='Meta'].copy()
snapchat = datorama[datorama['publisher']=='Snapchat'].copy()
tiktok = datorama[datorama['publisher']=='TikTok'].copy()
programmatic = datorama[datorama['channel']!='Social'].copy()

In [0]:
# Renaming SEM LOBs
col = sem['account_name']
sem['lob'] = np.where(col.str.contains('LBE'), 'LBE', 'Other')

# Renaming Meta LOBs
col = meta['line_of_business']
meta['lob'] = np.where(col.str.contains('LBE Brand'), 'LBE', 'Other')

# Renaming Snapchat LOBs
col = snapchat['line_of_business']
snapchat['lob'] = np.where(col.str.contains('LBE Brand'), 'LBE', 'Other')

# Renaming TikTok LOBs
col = tiktok['line_of_business']
tiktok['lob'] = np.where(col.str.contains('LBE Brand'), 'LBE', 'Other')

# Renaming Programmatic LOBs
col = programmatic['line_of_business']
programmatic['lob'] = np.where(col.str.contains('LBE Brand'), 'LBE', 'Other')

In [0]:
# Adding SEM objectives
col = sem['campaign']
sem['objective'] = np.select(
    [col.str.contains('Awareness|_AW|Generic_|Compete_|IMAX_New', na=False), 
     sem['account_name'].str.contains('Awareness', na=False),
     col.str.contains('Consideration|_CONS|_CN|Brand_S', na=False),
     sem['account_name'].str.contains('Consideration|B2B', na=False),
     col.str.contains('Conversion|_CVN|_CVS|001_CP2K790', na=False),
     sem['account_name'].str.contains('Conversion', na=False)],
    ['Awareness', 'Awareness', 'Consideration', 'Consideration', 'Conversion', 'Conversion'],
    default='Consideration')

# Changing Datorama objectives
cols = ['Awareness', 'Consideration', 'Conversion']
meta['objective'] = np.where(meta['objective'].isin(cols), meta['objective'], 'Consideration')
snapchat['objective'] = np.where(snapchat['objective'].isin(cols), snapchat['objective'], 'Unknown')
tiktok['objective'] = np.where(tiktok['objective'].isin(cols), tiktok['objective'], 'Unknown')
programmatic['objective'] = np.select(
    [programmatic['objective']=='Awareness', programmatic['objective']=='Consideration',
     programmatic['campaign_name'].str.contains('PDM|TRR'), programmatic['objective']=='Conversion'],
    ['Awareness', 'Consideration', 'Consideration', 'Conversion'], 
    default='Unknown')
pr['objective'] = 'Consideration'

In [0]:
# Setting channel column values
sem['channel'] = 'SEM'
meta['channel'] = 'Meta'
snapchat['channel'] = 'Snapchat'
tiktok['channel'] = 'TikTok'
programmatic['channel'] = 'Programmatic'

# Setting PR column values
pr = pr.assign(
    week=pr['week'],
    lob='PR',
    channel='PR',
    campaign_name='N/A',
    spend=pr['spend'],
    impressions=0,
    clicks=0,
    revenue=0
)

In [0]:
# Selecting columns
cols = [
    ['week', 'objective', 'lob', 
     'channel', 'campaign', 'cost', 
     'impr.', 'clicks'],
    ['week', 'objective', 'lob', 
     'channel', 'raw_campaign_name', 'investment', 
     'impressions', 'clicks'],
    ['week', 'objective', 'lob', 
     'channel', 'raw_campaign_name', 'investment', 
     'impressions', 'clicks'],
    ['week', 'objective', 'lob', 
     'channel', 'raw_campaign_name', 'investment', 
     'impressions', 'clicks'],
    ['week', 'objective', 'lob', 
     'channel', 'raw_campaign_name', 'investment', 
     'impressions', 'clicks']
    ]

sem = sem[cols[0]]
meta = meta[cols[1]]
snapchat = snapchat[cols[2]]
tiktok = tiktok[cols[3]]
programmatic = programmatic[cols[4]]

# Renaming columns
cols = [
    'week', 'objective', 'lob', 
    'channel', 'campaign_name', 'spend', 
    'impressions', 'clicks'
]

sem.columns = cols
meta.columns = cols
snapchat.columns = cols
tiktok.columns = cols
programmatic.columns = cols

In [0]:
# Combining data
spend_df = pd.concat([sem, meta, snapchat, tiktok, programmatic, pr[cols]]).fillna(0)
response_df = pd.concat([
    transactions.groupby('week').agg({'transactions': 'sum', 'revenue': 'sum'}),
    recroom_web.groupby('week').agg({'trr_web_visits': 'sum'}),
    playdium_web.groupby('week').agg({'pdm_web_visits': 'sum'})
], axis=1).reset_index()
control_df = pd.concat([mass_offers.groupby('week').agg({'mass_offer': 'max'})], axis=1).reset_index().fillna(0)

# Adding location types to spend data
col = spend_df['campaign_name']
spend_df['location_type'] = np.select(
    [spend_df['lob']=='Other',
     col.str.contains('Toronto|South Ed|Calgary|Deerfoot|Winnipeg|Brentwood|Burnaby|Vancouver'), 
     col.str.contains('HPL|High'), col.str.contains('West Ed|Square|Miss|London|Mason|John|Avalon|Barrie'), 
     col.str.contains('LPL|Low'), col.str.contains('Royalmount|Granville'), 
     col.str.contains('Brampton|Whitby|Dart'), col.str.contains('Fair'),
     col.str.contains('TRR|Rec Room'), col.str.contains('PDM|Playdium')],
    ['Other', 'HPL', 'HPL', 'LPL', 'LPL', 
     'NL - TRR', 'PDM', 'NL - PDM', 
     'Canada - TRR', 'Canada - PDM'],
    default='Multiple'
)

spend_df = spend_df[['week', 'objective', 'lob', 'location_type', 'channel', 
                     'campaign_name', 'spend', 'impressions', 'clicks']]

# Filtering data for trailing 2 years
start_date, end_date = pd.Timestamp('2023-03-31'), pd.Timestamp('2025-03-31')
spend_df = spend_df[spend_df['week'].between(start_date, end_date)]
response_df = response_df[response_df['week'].between(start_date, end_date)]
control_df = control_df[control_df['week'].between(start_date, end_date)]

# Renaming control columns
control_df.columns = ['week', 'mass_offer']

# Feature Engineering

In [0]:
# Defining holidays
holidays_by_name = {
    "New Year's Day": ['2023-01-01', '2024-01-01', '2025-01-01'],
    'Family Day': ['2023-02-20', '2024-02-19', '2025-02-17'],
    'Good Friday': ['2023-04-07', '2024-03-29', '2025-04-18'],
    'Easter Monday': ['2023-04-10', '2024-04-01', '2025-04-21'],
    'Victoria Day': ['2023-05-22', '2024-05-20', '2025-05-19'],
    'Canada Day': ['2023-07-01', '2024-07-01', '2025-07-01'],
    'Labour Day': ['2023-09-04', '2024-09-02', '2025-09-01'],
    'Thanksgiving': ['2023-10-09', '2024-10-14', '2025-10-13'],
    'Christmas Day': ['2023-12-25', '2024-12-25', '2025-12-25']
}

holidays = [pd.to_datetime(date) for date_list in holidays_by_name.values() for date in date_list]
holidays = pd.Series(holidays).dt.to_period('W-THU').dt.start_time

# Creating new columns
control_df['summer'] = np.where(control_df['week'].dt.month.isin([7, 8]), 1, 0)
control_df['holiday'] = np.where(control_df['week'].isin(holidays), 1, 0)

In [0]:
control_df.to_csv(directory + 'model_data/lbe_control_data.csv', index=False)
response_df.to_csv(directory + 'model_data/lbe_response_data.csv', index=False)
spend_df.to_csv(directory + 'model_data/lbe_spend_data.csv', index=False)